# Data Engineering et Machine Learning avec Snowflake

Ce notebook presente un pipeline complet dans Snowflake pour l'ingestion, la preparation des donnees, l'entrainement de modeles, l'inference et l'exposition du modele via Streamlit.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn
import xgboost

# Recuperer la session active Snowflake
session = get_active_session()

package_rows = [
    ("snowpark", "OK"),
    ("pandas", pd.__version__),
    ("numpy", np.__version__),
    ("scikit-learn", sklearn.__version__),
    ("matplotlib", matplotlib.__version__),
    ("seaborn", seaborn.__version__),
    ("xgboost", xgboost.__version__),
]

print("=" * 58)
print("Initialisation de l'environnement Snowflake")
print("=" * 58)
for package_name, package_version in package_rows:
    print(f"{package_name:<14} -> {package_version}")

print("")
print(f"Base active   : {session.get_current_database()}")
print(f"Schema actif  : {session.get_current_schema()}")
print(f"Warehouse     : {session.get_current_warehouse()}")
print("Environnement valide, vous pouvez lancer le pipeline.")


## 1. Ingestion des donnees

Dans cette etape, nous creons le format de fichier, le stage et la table brute pour charger le dataset JSON depuis S3.

In [ ]:
sql_statements = [
    """
    CREATE OR REPLACE FILE FORMAT HOUSE_JSON_FORMAT
      TYPE = 'JSON'
      STRIP_OUTER_ARRAY = TRUE
    """,
    """
    CREATE OR REPLACE STAGE HOUSE_STAGE
      URL = 's3://logbrain-datalake/datasets/house_price/'
      FILE_FORMAT = HOUSE_JSON_FORMAT
    """,
    """
    CREATE OR REPLACE TABLE HOUSE_PRICES_RAW (
      raw_data VARIANT
    )
    """,
    """
    COPY INTO HOUSE_PRICES_RAW
      FROM @HOUSE_STAGE/Housing_Price_Data.json
      FILE_FORMAT = HOUSE_JSON_FORMAT
    """
]

for statement in sql_statements:
    session.sql(statement).collect()

print("Ingestion terminee dans HOUSE_PRICES_RAW.")
print("Apercu du contenu brut :")
session.sql("SELECT raw_data FROM HOUSE_PRICES_RAW LIMIT 2").show()

print("Controle volumetrique apres ingestion :")
session.sql("""
SELECT
  COUNT(*) AS nb_lignes_chargees,
  COUNT(raw_data) AS nb_variants_non_null
FROM HOUSE_PRICES_RAW
""").show()


## 2. Creation de la table structuree

Les donnees brutes stockees en `VARIANT` sont converties en une table relationnelle typee, plus simple a explorer et a exploiter pour le machine learning.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE HOUSE_PRICES AS
SELECT
  raw_data:price::NUMBER             AS price,
  raw_data:area::NUMBER              AS area,
  raw_data:bedrooms::NUMBER          AS bedrooms,
  raw_data:bathrooms::NUMBER         AS bathrooms,
  raw_data:stories::NUMBER           AS stories,
  raw_data:mainroad::VARCHAR         AS mainroad,
  raw_data:guestroom::VARCHAR        AS guestroom,
  raw_data:basement::VARCHAR         AS basement,
  raw_data:hotwaterheating::VARCHAR  AS hotwaterheating,
  raw_data:airconditioning::VARCHAR  AS airconditioning,
  raw_data:parking::NUMBER           AS parking,
  raw_data:prefarea::VARCHAR         AS prefarea,
  raw_data:furnishingstatus::VARCHAR AS furnishingstatus
FROM HOUSE_PRICES_RAW
""").collect()

print("Table HOUSE_PRICES creee et typee.")
print("Extrait de la table structuree :")
session.sql("SELECT * FROM HOUSE_PRICES LIMIT 4").show()

print("Controle rapide des informations cles :")
session.sql("""
SELECT
  COUNT(*) AS nb_lignes,
  MIN(price) AS prix_min,
  MAX(price) AS prix_max,
  AVG(area) AS surface_moyenne
FROM HOUSE_PRICES
""").show()


## 3. Exploration initiale

Nous chargeons la table Snowflake dans un DataFrame Snowpark afin d'observer rapidement la volumetrie et les colonnes disponibles.

In [ ]:
# Charger la table dans un DataFrame Snowpark
df = session.table("HOUSE_PRICES")

row_count = df.count()
column_count = len(df.columns)
numeric_candidates = ["PRICE", "AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING"]
categorical_candidates = [col_name for col_name in df.columns if col_name not in numeric_candidates]

print(f"Volume du dataset : {row_count} lignes")
print(f"Nombre de colonnes : {column_count}")
print(f"Variables numeriques attendues : {numeric_candidates}")
print(f"Variables plutot categorielles : {categorical_candidates}")
print(f"Ordre complet des colonnes : {df.columns}")


## 4. Apercu des donnees

Cette section permet de visualiser les premieres lignes et les statistiques descriptives du dataset.

In [ ]:
print("=== Echantillon du dataset ===")
df.limit(5).show()

print("\n=== Resume statistique Snowpark ===")
df.describe().show()

print("\n=== Vue ciblee sur quelques colonnes metier ===")
session.sql("""
SELECT price, area, bedrooms, bathrooms, furnishingstatus
FROM HOUSE_PRICES
LIMIT 5
""").show()


## 5. Controle des valeurs manquantes

Avant toute preparation ou modelisation, nous verifions la presence eventuelle de valeurs nulles dans chaque colonne.

In [ ]:
from snowflake.snowpark.functions import col, count, when

# Calculer les valeurs nulles par colonne
null_counts = df.select([
    count(when(col(column_name).isNull(), column_name)).alias(column_name)
    for column_name in df.columns
])

print("=== Controle des valeurs nulles ===")
null_counts.show()
print("Lecture attendue : une ligne avec le nombre de nulls pour chaque colonne.")


## 6. Distribution de la variable cible

Nous etudions la distribution du prix des maisons avec un histogramme et un boxplot.

In [ ]:
import matplotlib.pyplot as plt

# Conversion vers Pandas pour la visualisation
df_pd = df.to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 5), facecolor="#f7f4ee")

axes[0].hist(df_pd["PRICE"], bins=26, color="#3d5a80", edgecolor="white", alpha=0.95)
axes[0].set_title("Repartition du prix des maisons")
axes[0].set_xlabel("Prix")
axes[0].set_ylabel("Nombre d'observations")
axes[0].grid(alpha=0.18)

axes[1].boxplot(
    df_pd["PRICE"],
    patch_artist=True,
    boxprops=dict(facecolor="#ee6c4d", color="#293241"),
    medianprops=dict(color="white", linewidth=2),
    whiskerprops=dict(color="#293241"),
    capprops=dict(color="#293241")
)
axes[1].set_title("Dispersion et valeurs extremes")
axes[1].set_ylabel("Prix")

plt.tight_layout()
plt.show()


## 7. Correlations entre variables numeriques

Une matrice de correlation permet d'identifier les relations lineaires entre les variables quantitatives.

In [ ]:
import seaborn as sns

num_cols = ["PRICE", "AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING"]
corr_matrix = df_pd[num_cols].corr()

plt.figure(figsize=(9, 6), facecolor="#f7f4ee")
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="crest",
    linewidths=0.6,
    square=True
)
plt.title("Matrice des correlations numeriques")
plt.tight_layout()
plt.show()


## 8. Analyse des variables categorielles

Nous observons la distribution des variables qualitatives du dataset.

In [ ]:
cat_cols = [
    "MAINROAD", "GUESTROOM", "BASEMENT",
    "HOTWATERHEATING", "AIRCONDITIONING",
    "PREFAREA", "FURNISHINGSTATUS"
]

color_cycle = ["#3d5a80", "#98c1d9", "#e0fbfc", "#ee6c4d", "#bc6c25", "#6c757d", "#84a98c"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8), facecolor="#f7f4ee")
axes = axes.flatten()

for i, col_name in enumerate(cat_cols):
    counts = df_pd[col_name].value_counts().sort_values(ascending=False)
    axes[i].bar(counts.index.astype(str), counts.values, color=color_cycle[i], edgecolor="white")
    axes[i].set_title(f"Distribution de {col_name.lower()}")
    axes[i].set_ylabel("Effectif")
    axes[i].tick_params(axis="x", rotation=18)

axes[-1].set_visible(False)

plt.suptitle("Vue d'ensemble des variables categorielles", fontsize=14)
plt.tight_layout()
plt.show()


## 9. Encodage des variables

Les colonnes categorielles utiles au modele sont converties en valeurs numeriques.

In [ ]:
import pandas as pd

df_pd = df.to_pandas()

binary_cols = ["MAINROAD", "GUESTROOM", "BASEMENT",
               "HOTWATERHEATING", "AIRCONDITIONING", "PREFAREA"]

for feature_name in binary_cols:
    df_pd[feature_name] = df_pd[feature_name].map({"yes": 1, "no": 0})

df_pd["FURNISHINGSTATUS"] = df_pd["FURNISHINGSTATUS"].map({
    "furnished": 2,
    "semi-furnished": 1,
    "unfurnished": 0
})

print("Encodage termine.")
print("Apercu des donnees encodees :")
print(df_pd.head())
print(f"\nTypes des colonnes apres transformation :\n{df_pd.dtypes}")


## 10. Separation des variables explicatives et de la cible

Le dataset est separe entre les features `X` et la variable cible `y`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Definir la cible
y = df_pd['PRICE']

# Conserver toutes les autres colonnes comme variables explicatives
X = df_pd.drop(columns=['PRICE'])

print(f"Matrice X : {X.shape}")
print(f"Vecteur cible y : {y.shape}")
print(f"\nVariables retenues : {list(X.columns)}")


## 11. Preparation train / test

Nous construisons les jeux d'entrainement et de test, puis nous appliquons une normalisation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train, 20% test
    random_state=42
)

# Standardisation des variables
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Dimensions de X_train : {X_train_scaled.shape}")
print(f"Dimensions de X_test  : {X_test_scaled.shape}")
print("\nRepartition du jeu de donnees :")
print(f"   Entrainement : {len(X_train)} lignes ({len(X_train)/len(X)*100:.0f}%)")
print(f"   Test         : {len(X_test)} lignes ({len(X_test)/len(X)*100:.0f}%)")


## 12. Entrainement des modeles

Trois modeles de regression sont entraines pour comparer leurs performances sur ce cas d'usage.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Definir les modeles a comparer
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42)
}

# Entrainer les modeles un par un
trained_models = {}
for model_name, estimator in models.items():
    estimator.fit(X_train_scaled, y_train)
    trained_models[model_name] = estimator
    print(f"Apprentissage termine pour : {model_name}")

print("\nLes trois modeles ont ete entraines.")


## 13. Evaluation des performances

Les modeles sont compares a l'aide de metriques adaptees a la regression : MAE, RMSE et R2.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = []

for model_name, estimator in trained_models.items():
    predictions = estimator.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Modele": model_name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2": round(r2, 4)
    })

    print(f"Resultats pour {model_name}")
    print(f"   - MAE  : {mae:,.0f}")
    print(f"   - RMSE : {rmse:,.0f}")
    print(f"   - R2   : {r2:.4f}\n")

# Construire le tableau recapitulatif
results_df = pd.DataFrame(results)
print("=== Tableau de comparaison des modeles ===")
print(results_df.to_string(index=False))


## 14. Visualisation comparative

Les metriques sont representees graphiquement pour faciliter l'interpretation des resultats.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor="#f7f4ee")
metrics = ["MAE", "RMSE", "R2"]
colors = ["#3d5a80", "#ee6c4d", "#84a98c"]

for i, metric in enumerate(metrics):
    axes[i].bar(results_df["Modele"], results_df[metric],
                color=colors[i], edgecolor="white")
    axes[i].set_title(f"Comparaison par {metric}")
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis="x", rotation=15)
    axes[i].grid(axis="y", alpha=0.15)

plt.suptitle("Lecture visuelle des metriques de performance", fontsize=14)
plt.tight_layout()
plt.show()


## 15. Optimisation par recherche d'hyperparametres

Un `GridSearchCV` est lance pour ameliorer les performances du modele XGBoost.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Parametres reduits pour rester leger en memoire
param_grid = {
    'xgb__n_estimators': [100, 200],
    'xgb__max_depth': [3, 5],
    'xgb__learning_rate': [0.1, 0.2],
}

pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(random_state=42))
])

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1,
    verbose=1
)

print("Lancement du GridSearchCV...")
grid_search.fit(X_train, y_train)

print(f"\nParametres optimaux : {grid_search.best_params_}")
print(f"Meilleur score R2 en validation croisee : {grid_search.best_score_:.4f}")


## 16. Evaluation du meilleur modele

Nous analysons les performances du modele optimise et nous les comparons au modele de base.

In [ ]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

mae_best = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best = r2_score(y_test, y_pred_best)

print("=== Evaluation du pipeline XGBoost optimise ===")
print(f"   MAE  : {mae_best:,.0f}")
print(f"   RMSE : {rmse_best:,.0f}")
print(f"   R2   : {r2_best:.4f}")

# Comparaison avec le modele XGBoost initial
xgb_base = trained_models['XGBoost']
y_pred_base = xgb_base.predict(X_test_scaled)
r2_base = r2_score(y_test, y_pred_base)

print("\nComparaison avant / apres optimisation :")
print(f"   XGBoost initial   -> R2 : {r2_base:.4f}")
print(f"   XGBoost optimise  -> R2 : {r2_best:.4f}")
improvement = (r2_best - r2_base) * 100
print(f"   Ecart observe     -> {improvement:+.2f}%")


## 17. Comparaison predictions / realite

Un nuage de points permet de visualiser la proximite entre les valeurs reelles et les valeurs predites.

In [ ]:
plt.figure(figsize=(8, 6), facecolor="#f7f4ee")
plt.scatter(y_test, y_pred_best, alpha=0.55, color="#3d5a80", edgecolors="white", s=46)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         linestyle="--", color="#ee6c4d", linewidth=2, label="Alignement parfait")
plt.xlabel("Prix reel")
plt.ylabel("Prix predit")
plt.title("Comparaison entre les valeurs reelles et predites")
plt.legend()
plt.tight_layout()
plt.show()


## 18. Enregistrement dans le Model Registry

Le meilleur modele est enregistre dans le Snowflake Model Registry avec ses metriques principales.

In [ ]:
from snowflake.ml.registry import Registry

# Creer le registry
registry = Registry(session=session)

model_name = "house_price_xgboost"
version_name = "v1"

# Enregistrer le meilleur modele (pipeline complet: scaler + XGBoost)
try:
    model_ref = registry.log_model(
        model=best_model,
        model_name=model_name,
        version_name=version_name,
        comment="Pipeline XGBoost optimise avec GridSearch",
        metrics={
            "r2": round(r2_best, 4),
            "mae": round(mae_best, 2),
            "rmse": round(rmse_best, 2)
        },
        sample_input_data=X_test.head(5)
    )
    print("Modele enregistre dans le Snowflake Model Registry.")
    print(f"   Nom du modele : {model_name}")
    print(f"   Version       : {version_name}")
    print(f"   R2            : {r2_best:.4f}")
except ValueError as err:
    if "already existed" in str(err):
        model_ref = registry.get_model(model_name).version(version_name)
        print("La version v1 existe deja dans le Model Registry.")
        print("La version existante est reutilisee pour la suite du notebook.")
        print(f"   Nom du modele : {model_name}")
        print(f"   Version       : {version_name}")
    else:
        raise


## 19. Verification du registry

Nous controlons que le modele enregistre est bien visible dans le registry.

In [ ]:
# Lister les modeles disponibles dans le registry
models_list = registry.show_models()
print("=== Contenu du Model Registry ===")
print(models_list)


## 20. Rechargement du modele

Le modele est recharge depuis le registry afin de preparer une etape d'inference.

In [ ]:
from snowflake.ml.registry import Registry
import pandas as pd

# Recharger le registry depuis la session
registry = Registry(session=session)

# Recuperer la version cible du modele
model_ref = registry.get_model("house_price_xgboost").version("v1")

print("Modele recupere depuis le registry.")


## 21. Inference sur de nouvelles donnees

Nous generons des predictions sur quelques maisons fictives pour valider le pipeline d'inference.

In [ ]:
# Construire un petit jeu d'exemples pour la prediction
new_houses = pd.DataFrame({
    'AREA': [200, 150, 300],
    'BEDROOMS': [3, 2, 4],
    'BATHROOMS': [2, 1, 3],
    'STORIES': [2, 1, 3],
    'MAINROAD': [1, 1, 0],
    'GUESTROOM': [0, 0, 1],
    'BASEMENT': [1, 0, 1],
    'HOTWATERHEATING': [0, 0, 1],
    'AIRCONDITIONING': [1, 0, 1],
    'PARKING': [2, 1, 3],
    'PREFAREA': [1, 0, 1],
    'FURNISHINGSTATUS': [2, 0, 1]
}).astype('int16')

# Le modele embarque deja les transformations necessaires
predictions = model_ref.run(new_houses, function_name="predict")

print("=== Resultats d'inference ===")
for idx, pred in enumerate(predictions['output_feature_0'], start=1):
    print(f"   Maison {idx} : {pred:,.0f} ")


## 22. Sauvegarde des predictions

Les resultats d'inference sont ecrits dans une table Snowflake dediee.

In [ ]:
# Ajouter les predictions au DataFrame source
new_houses['PREDICTED_PRICE'] = predictions['output_feature_0'].values

# Ecrire le resultat dans une table Snowflake
session.write_pandas(
    new_houses,
    table_name="HOUSE_PREDICTIONS",
    auto_create_table=True,
    overwrite=True
)

print("Les predictions ont ete sauvegardees dans HOUSE_PREDICTIONS.")

# Controler le contenu ecrit
session.table("HOUSE_PREDICTIONS").show()


## 23. Application Streamlit

Une interface Streamlit permet a un utilisateur metier de saisir des caracteristiques de bien immobilier et d'obtenir une estimation du prix.

In [ ]:
import streamlit as st
import pandas as pd
from streamlit.errors import StreamlitSetPageConfigMustBeFirstCommandError
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

try:
    st.set_page_config(
        page_title="PRIX//IMMO",
        page_icon="⬛",
        layout="wide",
        initial_sidebar_state="collapsed",
    )
except StreamlitSetPageConfigMustBeFirstCommandError:
    pass

# ─── Global CSS: dark brutalist / tech aesthetic ──────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Serif+Display&display=swap');

:root {
  --black:   #0a0a0a;
  --dark:    #111111;
  --panel:   #161616;
  --border:  #2a2a2a;
  --white:   #f0ede8;
  --dim:     #6b6b6b;
  --accent:  #e8ff47;
  --red:     #ff4444;
  --blue:    #4488ff;
}

html, body, [class*="css"] {
  font-family: 'Space Mono', monospace;
  background-color: var(--black) !important;
  color: var(--white);
}

section.main > div {
  background-color: var(--black) !important;
}

div.block-container {
  max-width: 1360px;
  padding: 0.75rem 1.5rem 2rem 1.5rem;
  background-color: var(--black) !important;
}

/* Sidebar */
div[data-testid="stSidebar"] {
  background-color: var(--dark) !important;
  border-right: 1px solid var(--border);
}
div[data-testid="stSidebar"] * { color: var(--dim) !important; }

/* Sliders */
div[data-testid="stSlider"] > div > div > div > div {
  background: var(--accent) !important;
}

/* Selectboxes */
div[data-testid="stSelectbox"] > div > div {
  background: var(--panel) !important;
  border: 1px solid var(--border) !important;
  color: var(--white) !important;
  border-radius: 0px !important;
  font-family: 'Space Mono', monospace;
}

/* Buttons */
div[data-testid="stButton"] > button {
  background: var(--accent) !important;
  color: var(--black) !important;
  border: none !important;
  border-radius: 0px !important;
  font-family: 'Space Mono', monospace !important;
  font-weight: 700 !important;
  font-size: 0.95rem !important;
  letter-spacing: 0.08em;
  padding: 0.75rem 1.5rem !important;
  transition: all 0.15s ease;
}
div[data-testid="stButton"] > button:hover {
  background: #ffffff !important;
  transform: translate(-2px, -2px);
  box-shadow: 4px 4px 0px var(--accent);
}

/* Metrics */
div[data-testid="stMetric"] {
  background: var(--panel);
  border: 1px solid var(--border);
  border-top: 3px solid var(--accent);
  padding: 1rem;
  border-radius: 0;
}
div[data-testid="stMetric"] label { color: var(--dim) !important; font-size: 0.72rem; }
div[data-testid="stMetric"] div[data-testid="stMetricValue"] { color: var(--white) !important; font-size: 1.4rem; }

/* Dataframe */
div[data-testid="stDataFrame"] { border: 1px solid var(--border); border-radius: 0; }

/* Labels */
label, .stSelectbox label, .stSlider label {
  color: var(--dim) !important;
  font-size: 0.72rem !important;
  letter-spacing: 0.1em;
  text-transform: uppercase;
}

/* Progress bar */
div[data-testid="stProgress"] > div > div {
  background: var(--accent) !important;
  border-radius: 0 !important;
}
div[data-testid="stProgress"] > div {
  background: var(--border) !important;
  border-radius: 0 !important;
}

hr { border-color: var(--border) !important; }
</style>
""", unsafe_allow_html=True)

# ─── Session ──────────────────────────────────────────────────────────────────
try:
    session = get_active_session()
except Exception:
    st.error("SESSION SNOWFLAKE NON DISPONIBLE — lancer depuis Snowflake Streamlit.")
    st.stop()

@st.cache_resource
def load_model():
    reg = Registry(session=session)
    return reg.get_model("house_price_xgboost").version("v1")

model_ref = load_model()

# ─── Header ──────────────────────────────────────────────────────────────────
st.markdown("""
<div style="
  border-bottom: 1px solid #2a2a2a;
  padding-bottom: 1.1rem;
  margin-bottom: 1.4rem;
  display: flex;
  align-items: flex-end;
  gap: 1.2rem;
">
  <div style="font-family:'DM Serif Display',serif; font-size:2.8rem; color:#f0ede8; line-height:1;">
    PRIX<span style="color:#e8ff47;">//</span>IMMO
  </div>
  <div style="color:#6b6b6b; font-size:0.72rem; letter-spacing:0.14em; padding-bottom:0.4rem;">
    MOTEUR D'ESTIMATION · SNOWFLAKE ML · XGBoost v1
  </div>
</div>
""", unsafe_allow_html=True)

# ─── Layout: 3 columns ───────────────────────────────────────────────────────
col_left, col_mid, col_right = st.columns([1, 1, 0.85], gap="large")

# ── LEFT: structural inputs ───────────────────────────────────────────────────
with col_left:
    st.markdown("""
    <div style="
      font-size:0.68rem; letter-spacing:0.18em; color:#6b6b6b;
      text-transform:uppercase; border-bottom:1px solid #2a2a2a;
      padding-bottom:0.4rem; margin-bottom:1rem;
    ">01 / Données structurelles</div>
    """, unsafe_allow_html=True)

    area      = st.slider("SURFACE (m²)", 50, 1000, 150, step=5)
    bedrooms  = st.slider("CHAMBRES", 1, 10, 3)
    bathrooms = st.slider("SALLES DE BAIN", 1, 5, 2)
    stories   = st.slider("ÉTAGES", 1, 5, 2)
    parking   = st.slider("PLACES DE PARKING", 0, 5, 1)

# ── MIDDLE: environment inputs ────────────────────────────────────────────────
with col_mid:
    st.markdown("""
    <div style="
      font-size:0.68rem; letter-spacing:0.18em; color:#6b6b6b;
      text-transform:uppercase; border-bottom:1px solid #2a2a2a;
      padding-bottom:0.4rem; margin-bottom:1rem;
    ">02 / Environnement & équipements</div>
    """, unsafe_allow_html=True)

    mainroad  = st.selectbox("ROUTE PRINCIPALE", ["yes", "no"])
    guestroom = st.selectbox("CHAMBRE D'AMIS", ["yes", "no"])
    basement  = st.selectbox("SOUS-SOL", ["yes", "no"])
    hotwater  = st.selectbox("CHAUFFAGE EAU CHAUDE", ["yes", "no"])
    aircon    = st.selectbox("CLIMATISATION", ["yes", "no"])
    prefarea  = st.selectbox("ZONE PRIVILÉGIÉE", ["yes", "no"])
    furnished = st.selectbox("AMEUBLEMENT", ["furnished", "semi-furnished", "unfurnished"])

# ── RIGHT: live profile card ──────────────────────────────────────────────────
with col_right:
    st.markdown("""
    <div style="
      font-size:0.68rem; letter-spacing:0.18em; color:#6b6b6b;
      text-transform:uppercase; border-bottom:1px solid #2a2a2a;
      padding-bottom:0.4rem; margin-bottom:1rem;
    ">03 / Profil du bien</div>
    """, unsafe_allow_html=True)

    yn = lambda v: 1 if v == "yes" else 0
    comfort = sum([yn(mainroad), yn(guestroom), yn(basement),
                   yn(hotwater), yn(aircon), yn(prefarea)])

    furnish_label = {"furnished": "MEUBLÉ", "semi-furnished": "SEMI-MEUBLÉ", "unfurnished": "NON MEUBLÉ"}
    segment = "COMPACT" if area < 120 else "FAMILIAL" if area < 260 else "PRESTIGE"
    segment_color = {"COMPACT": "#4488ff", "FAMILIAL": "#e8ff47", "PRESTIGE": "#ff4444"}[segment]

    st.markdown(f"""
    <div style="
      background: #161616;
      border: 1px solid #2a2a2a;
      border-left: 4px solid {segment_color};
      padding: 1.1rem 1rem;
      margin-bottom: 0.8rem;
    ">
      <div style="font-size:0.65rem;color:#6b6b6b;letter-spacing:0.15em;margin-bottom:0.3rem;">SEGMENT</div>
      <div style="font-size:1.5rem;color:{segment_color};font-weight:700;letter-spacing:0.08em;">{segment}</div>
      <div style="font-size:0.75rem;color:#6b6b6b;margin-top:0.2rem;">{area} m² · {bedrooms} ch. · {bathrooms} sdb</div>
    </div>

    <div style="
      background: #161616;
      border: 1px solid #2a2a2a;
      padding: 1rem;
      margin-bottom: 0.8rem;
    ">
      <div style="font-size:0.65rem;color:#6b6b6b;letter-spacing:0.15em;margin-bottom:0.6rem;">ÉQUIPEMENTS ACTIFS</div>
    """, unsafe_allow_html=True)

    tags_html = ""
    for label, val in [
        ("ROUTE", mainroad), ("AMIS", guestroom), ("CAVE", basement),
        ("EAU CH.", hotwater), ("CLIM", aircon), ("ZONE+", prefarea)
    ]:
        active = val == "yes"
        color  = "#e8ff47" if active else "#2a2a2a"
        tcolor = "#0a0a0a" if active else "#6b6b6b"
        tags_html += f"""<span style="
          display:inline-block; padding:0.18rem 0.5rem;
          background:{color}; color:{tcolor};
          font-size:0.65rem; letter-spacing:0.1em;
          margin:0.15rem 0.2rem 0.15rem 0;
        ">{label}</span>"""

    st.markdown(tags_html + "</div>", unsafe_allow_html=True)

    st.markdown(f"""
    <div style="font-size:0.65rem;color:#6b6b6b;letter-spacing:0.12em;margin-bottom:0.3rem;">
      INDICE DE CONFORT — {comfort}/6
    </div>
    """, unsafe_allow_html=True)
    st.progress(comfort / 6)

    st.markdown(f"""
    <div style="
      background:#161616; border:1px solid #2a2a2a;
      padding:0.7rem 1rem; margin-top:0.6rem;
      font-size:0.75rem; color:#6b6b6b;
    ">
      AMEUBLEMENT &nbsp;·&nbsp;
      <span style="color:#f0ede8;">{furnish_label[furnished]}</span>
      &nbsp;·&nbsp; PARKING {parking}
    </div>
    """, unsafe_allow_html=True)

# ─── Action bar ──────────────────────────────────────────────────────────────
st.markdown("<div style='height:1.4rem;'></div>", unsafe_allow_html=True)
st.markdown("<hr style='margin:0 0 1rem 0;'>", unsafe_allow_html=True)

btn_col, reset_col, _ = st.columns([0.22, 0.12, 0.66])
with btn_col:
    run = st.button("▶  LANCER L'ESTIMATION", use_container_width=True)
with reset_col:
    reset = st.button("✕  EFFACER", use_container_width=True)

if reset:
    st.session_state.pop("result", None)
    st.rerun()

# ─── Prediction ──────────────────────────────────────────────────────────────
furnish_map = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}

if run:
    input_df = pd.DataFrame([{
        "AREA":             area,
        "BEDROOMS":         bedrooms,
        "BATHROOMS":        bathrooms,
        "STORIES":          stories,
        "MAINROAD":         yn(mainroad),
        "GUESTROOM":        yn(guestroom),
        "BASEMENT":         yn(basement),
        "HOTWATERHEATING":  yn(hotwater),
        "AIRCONDITIONING":  yn(aircon),
        "PARKING":          parking,
        "PREFAREA":         yn(prefarea),
        "FURNISHINGSTATUS": furnish_map[furnished],
    }]).astype("int16")

    with st.spinner(""):
        result   = model_ref.run(input_df, function_name="predict")
        price    = float(result["output_feature_0"].values[0])
        ppsm     = price / max(area, 1)

    st.session_state["result"] = {
        "price": price, "ppsm": ppsm,
        "input_df": input_df, "comfort": comfort, "segment": segment
    }

# ─── Result display ───────────────────────────────────────────────────────────
if "result" in st.session_state:
    r = st.session_state["result"]
    price, ppsm = r["price"], r["ppsm"]

    # Big result banner
    st.markdown(f"""
    <div style="
      background: #0a0a0a;
      border: 1px solid #2a2a2a;
      border-top: 4px solid #e8ff47;
      padding: 1.6rem 1.8rem;
      margin: 1.2rem 0 1rem 0;
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 1rem;
    ">
      <div>
        <div style="font-size:0.65rem;color:#6b6b6b;letter-spacing:0.18em;margin-bottom:0.3rem;">
          VALEUR ESTIMÉE
        </div>
        <div style="
          font-family:'DM Serif Display',serif;
          font-size:3.2rem;
          color:#e8ff47;
          line-height:1;
          letter-spacing:-0.01em;
        ">{price:,.0f} <span style="font-size:1.4rem;color:#6b6b6b;">EUR</span></div>
      </div>
      <div style="text-align:right;">
        <div style="font-size:0.65rem;color:#6b6b6b;letter-spacing:0.18em;margin-bottom:0.3rem;">AU M²</div>
        <div style="font-size:1.8rem;color:#f0ede8;">{ppsm:,.0f} <span style="font-size:0.9rem;color:#6b6b6b;">EUR/m²</span></div>
        <div style="font-size:0.68rem;color:#6b6b6b;margin-top:0.4rem;">
          SEGMENT <span style="color:#f0ede8;">{r['segment']}</span>
          &nbsp;·&nbsp;
          CONFORT <span style="color:#f0ede8;">{r['comfort']}/6</span>
        </div>
      </div>
    </div>
    """, unsafe_allow_html=True)

    # Metrics row
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("PRIX ESTIMÉ",  f"{price:,.0f} €")
    m2.metric("PRIX AU M²",   f"{ppsm:,.0f} €/m²")
    m3.metric("SURFACE",      f"{area} m²")
    m4.metric("CONFORT",      f"{r['comfort']}/6")

    # Input vector
    st.markdown("""
    <div style="
      font-size:0.65rem;letter-spacing:0.18em;color:#6b6b6b;
      text-transform:uppercase;border-bottom:1px solid #2a2a2a;
      padding-bottom:0.35rem;margin:1.2rem 0 0.6rem 0;
    ">VECTEUR D'ENTRÉE TRANSMIS AU MODÈLE</div>
    """, unsafe_allow_html=True)
    st.dataframe(
        r["input_df"].T.rename(columns={0: "VALEUR"}),
        use_container_width=True
    )

    st.markdown("""
    <div style="
      font-size:0.68rem; color:#6b6b6b; letter-spacing:0.08em;
      border-top:1px solid #2a2a2a; padding-top:0.7rem; margin-top:0.8rem;
    ">
      ✓ INFERENCE EXÉCUTÉE ·
      MODÈLE : house_price_xgboost ·
      VERSION : v1 ·
      PIPELINE : StandardScaler + XGBoost ·
      SORTIE : output_feature_0
    </div>
    """, unsafe_allow_html=True)
